In [24]:
text = """
인공지능의 단계를 묘사할 때 흔히 사용되는 용어인 약인공지능weak AI과 강인공지능strong AI은 1980년에 존 설John R. Searle, (1932~2025) 교수가 그 유명한 중국어 방 논변을 제안하면서 최초로 사용한 개념이다.

다른 곳의 문서를 보면 인간의 마음을 복잡한 정보처리로 구현한 것을 강한 인공지능, 단순히 인간의 능력 일부를 시뮬레이션 하거나 그런 작업을 목적으로 하는 것을 약한 인공지능이라고 편리하게 설명한다. 그러나 좀 더 정확하게 설명하자면 존 설 교수의 본래 의도는 인간의 마음을 컴퓨터 소프트웨어와 같은 관점으로 보고 행하는 인공지능 연구를 "강한 인공지능 연구", 반대로 인간의 마음과는 별개로 단지 유용한 도구의 개발을 위해 행하는 인공지능 연구를 "약한 인공지능 연구"로 정의하여 철학적 관점에서 인간의 마음을 컴퓨터와 소프트웨어와 같이 보는 연구를 비판하고자 한 것이다.

아무튼 저 개념이 묘하게 현재의 인공지능 개발현황과 의미가 적절히 맞아떨어지다보니 내용을 살짝 비틀어서 대중들에게 알려지게 되었는데 각 연구의 결과물을 그대로 대입하면 된다. 즉,

강인공지능 = 인간을 완벽하게 모방한 인공지능이고
약인공지능 = 유용한 도구로서 설계된 인공지능이라고 대중들에게 많이 인식되어지고 있으며, 현재까지 나온 인공지능은 모두 약인공지능으로 구분하고 있다.

물론 인공지능의 역사로 볼 때 '인공지능'이라는 단어 자체가 처음부터 적절한 단어는 아니었기 때문에 약간의 왜곡이 들어간 해석이라고 보기도 하며, 인공지능 모델의 연구를 보면 인간의 사고과정을 모사할 수 있는 것이 목표이기는 하지만 단순하게 인간을 모방한 척도 자체가 인공지능을 평가하는 척도는 아니다.

실제로 인공지능 모델 연구에서 AGI같은 용어를 제외하면, 강인공지능, 약인공지능이라는 표현이 거의 쓰이지는 않는다. 1980년 이 용어가 처음 등장할 때만 해도 겨우 역전파 알고리즘, CNN 정도가 단순 이론적으로만 구상되던 시기였기 때문에, 인공지능의 이론적 배경이 들어갔다기 보다는 인공지능의 초기 역사에서 미래를 상상하며 약간의 SF 향이 첨가된 용어라고 이해하면 쉽다.
"""

In [25]:
import torch

# 단어 단위 토큰화
tokens = text.split()

# vocab 생성
vocab = sorted(list(set(tokens)))
stoi = {w:i for i,w in enumerate(vocab)}
itos = {i:w for w,i in stoi.items()}

vocab_size = len(vocab)

def encode(s):
    return [stoi[w] for w in s.split()]

def decode(l):
    return " ".join([itos[i] for i in l])

In [26]:
seq_len = 5

data = encode(text)

X, y = [], []

for i in range(len(data) - seq_len):
    X.append(data[i:i+seq_len])
    y.append(data[i+1:i+seq_len+1])

X = torch.tensor(X)
y = torch.tensor(y)

In [27]:
def causal_mask(size):
    return torch.tril(torch.ones(size, size))

In [28]:
import torch.nn as nn
import torch.nn.functional as F

class SelfAttention(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.q = nn.Linear(d_model, d_model)
        self.k = nn.Linear(d_model, d_model)
        self.v = nn.Linear(d_model, d_model)

    def forward(self, x):
        B, T, D = x.shape

        Q = self.q(x)
        K = self.k(x)
        V = self.v(x)

        scores = Q @ K.transpose(-2, -1) / (D ** 0.5)

        mask = causal_mask(T).to(x.device)
        scores = scores.masked_fill(mask == 0, float('-inf'))

        attn = F.softmax(scores, dim=-1)

        out = attn @ V
        return out

In [29]:
class Block(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.attn = SelfAttention(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_model*4),
            nn.ReLU(),
            nn.Linear(d_model*4, d_model)
        )
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)

    def forward(self, x):
        x = self.ln1(x + self.attn(x))
        x = self.ln2(x + self.ffn(x))
        return x

In [30]:
class GPT(nn.Module):
    def __init__(self, vocab_size, d_model, seq_len):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(seq_len, d_model)

        self.blocks = nn.Sequential(
            Block(d_model),
            Block(d_model)
        )

        self.ln = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        B, T = x.shape

        tok = self.token_emb(x)
        pos = self.pos_emb(torch.arange(T, device=x.device))

        x = tok + pos
        x = self.blocks(x)
        x = self.ln(x)

        return self.head(x)

In [31]:
model = GPT(vocab_size, d_model=64, seq_len=seq_len)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

for epoch in range(300):
    optimizer.zero_grad()

    out = model(X)

    loss = criterion(out.view(-1, vocab_size), y.view(-1))

    loss.backward()
    optimizer.step()

    if epoch % 50 == 0:
        print(epoch, loss.item())

0 5.483780860900879
50 1.316713809967041
100 0.2182779163122177
150 0.10855959355831146
200 0.0824807733297348
250 0.07139372825622559


In [33]:
def generate(model, start, max_len=10):
    model.eval()

    tokens = encode(start)

    for _ in range(max_len):
        x = torch.tensor(tokens[-seq_len:]).unsqueeze(0)

        out = model(x)
        next_token = out[0, -1].argmax().item()

        tokens.append(next_token)

    return decode(tokens)

In [34]:
print(generate(model, "단계를", 10))

단계를 묘사할 때 흔히 사용되는 용어인 약인공지능weak AI과 강인공지능strong AI은 1980년에


In [35]:
# 단어 임베딩 행렬
print(model.token_emb.weight)
print(model.token_emb.weight.shape) # 16개 단어를 64차원으로 표현

Parameter containing:
tensor([[-0.9552, -0.8210, -0.3597,  ...,  0.1396,  0.5482, -0.4685],
        [-0.3834,  1.2830,  0.1430,  ...,  0.1056,  0.7391, -0.4795],
        [-0.7055, -0.0177, -0.0901,  ...,  0.1653,  0.5003,  1.1342],
        ...,
        [ 0.6005, -0.1575, -0.2661,  ...,  2.1642,  0.9631,  2.2487],
        [ 1.4920, -1.6679,  0.3745,  ..., -0.0691,  0.4380, -0.6295],
        [ 0.1703,  3.2082, -0.5866,  ..., -1.2177,  0.9148,  0.5663]],
       requires_grad=True)
torch.Size([205, 64])


In [23]:
# 개선 할점
# 조사를 자연스럽게 처리할수있도록
# 문장의 시작과 끝을 알수있도록 단어사전에 <SOS> <EOS> 넣고 학습하기
# 파일 여러개 띄워놓고 파일 단위로 학습하기 (토크나이져, 모델, 대화모드 출력부분, 성능테스트(평가), 공통함수 등)
# 토크나이저나 모델들도 다 내가 구현하기(독자, 원천기술)